## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

### Loading datasets

In [2]:
train = pd.read_csv("./dataset/train.csv")
public_test = pd.read_csv("./dataset/public_test.csv")
private_test = pd.read_csv("./dataset/private_test.csv")

### Phase 0 - Inspecting datasets

In [3]:
# Checking shapes:

print("Training dataset shape: ", train.shape)
print("Public Test shape: ", public_test.shape)
print("Private Test shape: ", private_test.shape)

Training dataset shape:  (10000, 14)
Public Test shape:  (3000, 14)
Private Test shape:  (3000, 13)


##### Why do we need to analyse `.shape`?
Because it will help me to know how much data I actually have and what kind of problem scale is this.
From this I can infer that I have `10000` 'Training data' and `3000` 'Public test data' which is enough to evaluate experiments.
Private test has one fewer column -> `Converted` column is hidden; Which exactly is what I am supposed to predict.

In [4]:
# Inspecting columns:
train.columns

Index(['User_ID', 'Age', 'Income', 'City_Tier', 'Device_Type',
       'Traffic_Source', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site',
       'Previous_Purchases', 'Discount_Seen', 'Browser_Version',
       'Campaign_Code', 'Converted'],
      dtype='object')

In [5]:
# Peeking into rows
train.head()

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


In [6]:
# to check target distribution
train["Converted"].value_counts()

Converted
0    6913
1    3087
Name: count, dtype: int64

In [7]:
# inspecting data types
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             10000 non-null  int64  
 1   Age                 8520 non-null   float64
 2   Income              9016 non-null   float64
 3   City_Tier           10000 non-null  int64  
 4   Device_Type         10000 non-null  object 
 5   Traffic_Source      10000 non-null  object 
 6   Pages_Viewed        10000 non-null  int64  
 7   Products_Viewed     10000 non-null  int64  
 8   Time_On_Site        8152 non-null   float64
 9   Previous_Purchases  10000 non-null  int64  
 10  Discount_Seen       10000 non-null  int64  
 11  Browser_Version     10000 non-null  int64  
 12  Campaign_Code       10000 non-null  int64  
 13  Converted           10000 non-null  int64  
dtypes: float64(3), int64(9), object(2)
memory usage: 1.1+ MB


In [8]:
# Check missing values column wise
print("Missing values in Train data: \n", train.isnull().sum())
print("\nMissing values in Public Test data: \n", public_test.isnull().sum())
print("\nMissing values in Private Test data: \n", private_test.isnull().sum())

Missing values in Train data: 
 User_ID                  0
Age                   1480
Income                 984
City_Tier                0
Device_Type              0
Traffic_Source           0
Pages_Viewed             0
Products_Viewed          0
Time_On_Site          1848
Previous_Purchases       0
Discount_Seen            0
Browser_Version          0
Campaign_Code            0
Converted                0
dtype: int64

Missing values in Public Test data: 
 User_ID                 0
Age                   472
Income                301
City_Tier               0
Device_Type             0
Traffic_Source          0
Pages_Viewed            0
Products_Viewed         0
Time_On_Site          568
Previous_Purchases      0
Discount_Seen           0
Browser_Version         0
Campaign_Code           0
Converted               0
dtype: int64

Missing values in Private Test data: 
 User_ID                 0
Age                   461
Income                302
City_Tier               0
Device_Type      

In [9]:
train.dtypes

User_ID                 int64
Age                   float64
Income                float64
City_Tier               int64
Device_Type            object
Traffic_Source         object
Pages_Viewed            int64
Products_Viewed         int64
Time_On_Site          float64
Previous_Purchases      int64
Discount_Seen           int64
Browser_Version         int64
Campaign_Code           int64
Converted               int64
dtype: object

In [10]:
train.describe()

,User_ID,Age,Income,City_Tier,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
count,10000.00000,8520.000000,9016.000000,10000.000000,10000.00000,10000.000000,8152.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,41.457746,69961.772797,1.933400,15.60850,15.219300,13.667241,2.972500,0.540700,12.473900,5511.162200,0.30870
std,2886.89568,13.770164,24790.673822,0.737712,8.62346,8.927563,19.438244,1.738691,0.498366,6.892856,2596.933802,0.46198
min,1.00000,18.000000,12000.000000,1.000000,1.00000,1.000000,0.800000,0.000000,0.000000,1.000000,1000.000000,0.00000
25%,2500.75000,29.000000,52294.644359,1.000000,8.00000,8.000000,7.640000,2.000000,0.000000,7.000000,3251.750000,0.00000
50%,5000.50000,41.000000,70171.613672,2.000000,16.00000,15.000000,11.145000,3.000000,1.000000,12.000000,5515.500000,0.00000
75%,7500.25000,53.000000,86907.747154,2.000000,23.00000,23.000000,15.630000,4.000000,1.000000,18.000000,7759.000000,1.00000
max,10000.00000,65.000000,161687.774167,3.000000,30.00000,37.000000,607.390000,12.000000,1.000000,24.000000,9998.000000,1.00000


#### What I saw was in `Time_On_Site` column `50% = 11.14` but `max = 607`? -> Candidate outlier
So Phase 0 conclusion:

I found:

- Missing values -> `Age`, `Income`, `Time_On_Site`
- Mild class imbalance -> `~69/31`
- Categorical columns exist (the ones with `object` data type)
- Possible outlier -> Time_On_Site
- Data scale looks healthy

### Phase 1A - Exploratory Data Analysis (EDA)

In [11]:
train['Converted'].mean()

np.float64(0.3087)

In [12]:
train.groupby("Device_Type")["Converted"].mean()

Device_Type
Desktop    0.330906
Mobile     0.300629
Tablet     0.306468
Name: Converted, dtype: float64

In [13]:
train.groupby("Traffic_Source")["Converted"].mean()

Traffic_Source
Email           0.349359
Organic         0.291762
Paid Ads        0.277123
Referral        0.381181
Social Media    0.303061
Name: Converted, dtype: float64

##### `Traffic_Source` may be more informative than `Device_Type`
| Traffic Source | Conversion Rate | Device Type | Conversion Rate |
| :--- | :--- | :--- | :--- |
| **Referral** | 38.12% | **Desktop** | 33.09% |
| **Email** | 34.94% | **Tablet** | 30.65% |
| **Social Media** | 30.31% | **Mobile** | 30.06% |
| **Organic** | 29.18% | | |
| **Paid Ads** | 27.71% | | |

In [14]:
train["Traffic_Source"].value_counts()

Traffic_Source
Organic         3071
Paid Ads        2544
Social Media    1993
Referral        1456
Email            936
Name: count, dtype: int64

Referral → 38%
Referral users = 1456

Email → 35%
but only 936 users

Still okay, but smaller evidence.
This is our first proper EDA-style thought:
A feature can look strong, but we should also check how much data supports that observation.

In [15]:
train.groupby("Converted")["Time_On_Site"].mean()

Converted
0    12.597556
1    15.925828
Name: Time_On_Site, dtype: float64

Converted
- 0   -> 12.597556
- 1   -> 15.925828
- So, users who converted stayed longer on site on average.

In [16]:
train.groupby("Converted")["Pages_Viewed"].mean()

Converted
0    13.802546
1    19.652737
Name: Pages_Viewed, dtype: float64

Converted
- 0 : 13.802546
- 1 : 19.652737
- So converted viewed more page on average.

In [17]:
train.groupby("Converted")["Products_Viewed"].mean()

Converted
0    13.360480
1    19.381924
Name: Products_Viewed, dtype: float64

So converted users tend to view more products too.
- 0 : 13.360480
- 1 : 19.381924

That suggests:
- engagement metrics may contain useful predictive signal.
Meaning:
- Users who engage more seem more likely to convert.

Again: Not proof. Only relationship.

In [18]:
train[["Pages_Viewed", "Products_Viewed"]].corr()

,Pages_Viewed,Products_Viewed
Pages_Viewed,1.000000,0.945833
Products_Viewed,0.945833,1.000000


When two features look similar:

- `Pages_Viewed`
- `Products_Viewed`

we should ask: Are they basically measuring the same thing?

|                 | Pages_Viewed | Products_Viewed |
| --------------- | -----------: | --------------: |
| Pages_Viewed    |         1.00 |           0.946 |
| Products_Viewed |        0.946 |            1.00 |

First understand the diagonal:
- Pages ↔ Pages = 1
- Products ↔ Products = 1
Every feature perfectly matches itself; IGNORE IT.

The interesting value is: `0.946`.  
That is very high.

Question: What does that mean?

Imagine:  
`User views more pages.`  
↓  
`also views more products.`  

These two move together a lot.  
So: `Pages_Viewed` and `Products_Viewed` are strongly positively correlated

What we know so far from EDA:

- Device_Type → weak difference
- Traffic_Source → stronger difference
- Converted users stay longer
- Converted users view more pages
- Converted users view more products
- Pages and Products are highly related

In [19]:
train.groupby("Converted")["Previous_Purchases"].mean()
# People who converted had slightly more previous purchases on average. Difference: ~0.36 purchases

Converted
0    2.859974
1    3.224490
Name: Previous_Purchases, dtype: float64

Previous_Purchases may contain signal,
but weaker than engagement metrics

In [20]:
train.groupby("Discount_Seen")["Converted"].mean()

Discount_Seen
0    0.255171
1    0.354171
Name: Converted, dtype: float64

Users who saw discounts converted more often.

Discoveries in EDA:  
| Feature            | Observation             |
| ------------------ | ----------------------- |
| Device_Type        | weak relationship       |
| Traffic_Source     | noticeable relationship |
| Time_On_Site       | converters stay longer  |
| Pages_Viewed       | converters browse more  |
| Products_Viewed    | converters inspect more |
| Previous_Purchases | slight relationship     |
| Discount_Seen      | stronger relationship   |


In [21]:
train.groupby("City_Tier")["Converted"].mean()

# City_Tier does not immediately look very strong

City_Tier
1    0.302991
2    0.320337
3    0.294191
Name: Converted, dtype: float64

In [22]:
train.groupby("Converted")["Income"].mean()

Converted
0    69134.659095
1    71771.595850
Name: Income, dtype: float64

In [23]:
train.groupby("Converted")["Age"].mean()

Converted
0    41.548475
1    41.253435
Name: Age, dtype: float64

In [24]:
train.corr(numeric_only=True)["Converted"].sort_values()

Age                  -0.009888
Campaign_Code        -0.008329
City_Tier            -0.005108
Browser_Version      -0.001380
User_ID               0.001201
Income                0.049356
Time_On_Site          0.079968
Previous_Purchases    0.096854
Discount_Seen         0.106797
Products_Viewed       0.311595
Pages_Viewed          0.313409
Converted             1.000000
Name: Converted, dtype: float64

| Feature            | Correlation |
| ------------------ | ----------: |
| Pages_Viewed       |       0.313 |
| Products_Viewed    |       0.312 |
| Discount_Seen      |       0.107 |
| Previous_Purchases |       0.097 |
| Time_On_Site       |       0.080 |
| Income             |       0.049 |
| Age                |      −0.010 |

Earlier:  
- Pages looked strong
- Products looked strong
- Age looked weak

This correlation agrees.

Two important lessons:
1. Bigger magnitude ≠ more important automatically.  
   Correlation only measures linear relationship.  
   A feature with low correlation may still help a model.

2. Correlation does NOT mean causation

### Phase 2 - Preprocessing

1. I found missing values in:  
| Column       | Missing |
| ------------ | ------: |
| Age          |    1480 |
| Income       |     984 |
| Time_On_Site |    1848 |

2. For missing values there are generally 3 choices:
   - Drop Rows : Delete rows with missing values
   - Fill with statistic : median, mean, etc..
   - Predict missing values : using other features

3. But,
   - Missing rows are not additive, but worst case: `1480 + 984 + 1848` | Potentially thousands affected.
   - Dataset: `10000` rows

     So, throwing away maybe `~15–30%` of training data immediately? That’s a lot; Probably risky.


4. So my current hypothesis: `Dropping rows is unlikely to be my first choice`.  
   That leaves : `Fill values`.  
   Question : Fill with what? -> What are reasonable replacement values?

In [25]:
train[["Age","Income","Time_On_Site"]].median()

Age                41.000000
Income          70171.613672
Time_On_Site       11.145000
dtype: float64

Median became my first PreProcessing candidate because earlier I had noticed an outlier in `Time_On_Site` which pretty sure will pull `mean`. Therefore, `Median` in this case is more resilient.

In [26]:
train["Age"].isnull().groupby(train["Converted"]).mean()

Converted
0    0.146536
1    0.151280
Name: Age, dtype: float64

In [27]:
train["Time_On_Site"].isnull().groupby(train["Converted"]).mean()

Converted
0    0.199769
1    0.151280
Name: Time_On_Site, dtype: float64

In [28]:
train["Income"].isnull().groupby(train["Converted"]).mean()

Converted
0    0.104875
1    0.083900
Name: Income, dtype: float64

Let's look at the gap between the two classes. The difference is roughly $2.1\%$ ($10.49\% - 8.39\%$).While this gap isn't quite as massive as the $4.8\%$ delta we discovered for Time_On_Site, it is still significantly larger than the tiny $0.5\%$ noise we observed in Age.

If we simply overwrite all NaN values in Income with the median, we destroy this subtle $2.1\%$ behavioral difference. The machine learning model will never get to see that a blank income field slightly skews towards a non-converter.Our Strategy Update:  
To preserve this signal without breaking our downstream model pipelines, we will treat Income exactly like we treated Time_On_Site. We will use a Missing Indicator Feature alongside our imputation.

In [29]:
train["Time_On_Site_missing"] = train["Time_On_Site"].isnull().astype(int)

In [30]:
train.head()

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted,Time_On_Site_missing
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0,1
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0,0


In [31]:
time_on_site_median = train["Time_On_Site"].median()
train["Time_On_Site"] = train["Time_On_Site"].fillna(time_on_site_median)

#### Creating the Missing Indicator Column for `Income`

In [32]:
train["Income_missing"] = train["Income"].isnull().astype(int)

In [33]:
income_median = train["Income"].median()
train["Income"] = train["Income"].fillna(income_median)

In [34]:
train.head(10)

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted,Time_On_Site_missing,Income_missing
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.610,3,0,11,2418,0,0,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.630,2,0,14,1213,0,0,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.250,5,0,5,2849,0,0,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,11.145,1,1,19,7610,0,1,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.920,1,0,5,9261,0,0,0
5,6,40.0,70171.613672,2,Mobile,Paid Ads,27,30,17.100,1,0,16,8638,0,0,1
6,7,21.0,64908.075218,2,Mobile,Email,30,26,16.590,1,0,11,7266,0,0,0
7,8,35.0,114200.619715,1,Desktop,Paid Ads,7,7,2.350,1,1,23,6252,1,0,0
8,9,48.0,57021.611431,3,Mobile,Paid Ads,6,9,14.490,6,1,20,7480,1,0,0
9,10,35.0,66561.117857,1,Desktop,Organic,1,7,160.240,6,0,20,7906,1,0,0


In [35]:
train.isnull().sum()

User_ID                    0
Age                     1480
Income                     0
City_Tier                  0
Device_Type                0
Traffic_Source             0
Pages_Viewed               0
Products_Viewed            0
Time_On_Site               0
Previous_Purchases         0
Discount_Seen              0
Browser_Version            0
Campaign_Code              0
Converted                  0
Time_On_Site_missing       0
Income_missing             0
dtype: int64

In [36]:
age_median = train["Age"].median()
train["Age"] = train["Age"].fillna(age_median)

In [37]:
train["Device_Type"].unique()

array(['Mobile', 'Desktop', 'Tablet'], dtype=object)

In [38]:
train["Traffic_Source"].unique()

array(['Organic', 'Social Media', 'Referral', 'Email', 'Paid Ads'],
      dtype=object)

In [39]:
pd.get_dummies(train["Device_Type"]).astype(int).head(10)

,Desktop,Mobile,Tablet
0,0,1,0
1,0,1,0
2,0,1,0
3,0,1,0
4,0,1,0
5,0,1,0
6,0,1,0
7,1,0,0
8,0,1,0
9,1,0,0


In [40]:
categorical_columns = ["Device_Type", "Traffic_Source"]
train = pd.get_dummies(train, columns=categorical_columns, dtype=int)

In [41]:
train.head(10)

,User_ID,Age,Income,City_Tier,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,...,Time_On_Site_missing,Income_missing,Device_Type_Desktop,Device_Type_Mobile,Device_Type_Tablet,Traffic_Source_Email,Traffic_Source_Organic,Traffic_Source_Paid Ads,Traffic_Source_Referral,Traffic_Source_Social Media
0,1,58.0,103593.708812,2,5,4,9.610,3,0,11,...,0,0,0,1,0,0,1,0,0,0
1,2,26.0,36451.716984,2,11,3,17.630,2,0,14,...,0,0,0,1,0,0,0,0,0,1
2,3,19.0,30511.228700,3,1,1,13.250,5,0,5,...,0,0,0,1,0,0,0,0,1,0
3,4,48.0,87789.172342,3,14,12,11.145,1,1,19,...,1,0,0,1,0,1,0,0,0,0
4,5,35.0,105229.249067,2,14,21,16.920,1,0,5,...,0,0,0,1,0,0,0,0,0,1
5,6,40.0,70171.613672,2,27,30,17.100,1,0,16,...,0,1,0,1,0,0,0,1,0,0
6,7,21.0,64908.075218,2,30,26,16.590,1,0,11,...,0,0,0,1,0,1,0,0,0,0
7,8,35.0,114200.619715,1,7,7,2.350,1,1,23,...,0,0,1,0,0,0,0,1,0,0
8,9,48.0,57021.611431,3,6,9,14.490,6,1,20,...,0,0,0,1,0,0,0,1,0,0
9,10,35.0,66561.117857,1,1,7,160.240,6,0,20,...,0,0,1,0,0,0,1,0,0,0


In [42]:
# 1. Isolate the features (X) and the target variable (y)
X = train.drop(columns=["User_ID", "Converted"])
y = train["Converted"]

### Phase 3 - Data Splitting

In [43]:
# 2. Split the completely raw data first (keeps validation totally secret)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

### Phase 4 - Model Building and Baseline Evaluation

In [47]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Combine training and validation back into the 100% full train dataset
full_train_clean = pd.concat([X_train, X_val], axis=0).copy()
full_y_train = pd.concat([y_train, y_val], axis=0).copy()
public_test_clean = public_test.copy()

# 2. Add the missingness indicator column to both
full_train_clean['Time_On_Site_missing'] = full_train_clean['Time_On_Site'].isnull().astype(int)
public_test_clean['Time_On_Site_missing'] = public_test_clean['Time_On_Site'].isnull().astype(int)

# 3. Clean up missing values using the full training medians
winning_features = ['Income', 'Pages_Viewed', 'Discount_Seen', 'Previous_Purchases', 'Time_On_Site', 'City_Tier']
for col in winning_features:
    median_val = full_train_clean[col].median()
    full_train_clean[col] = full_train_clean[col].fillna(median_val)
    public_test_clean[col] = public_test_clean[col].fillna(median_val)

# 4. Clamp continuous outliers at the 95th percentile of the full training data
for col in ['Time_On_Site', 'Income', 'Pages_Viewed']:
    upper_limit = full_train_clean[col].quantile(0.95)
    full_train_clean[col] = np.where(full_train_clean[col] > upper_limit, upper_limit, full_train_clean[col])
    public_test_clean[col] = np.where(public_test_clean[col] > upper_limit, upper_limit, public_test_clean[col])

# 5. Isolate and scale the features
scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(full_train_clean[winning_features])
X_public_scaled_full = scaler_full.transform(public_test_clean[winning_features])

# 6. Train Logistic Regression on 100% of data and test on public_test
final_lr = LogisticRegression(random_state=42, max_iter=1000)
final_lr.fit(X_train_scaled_full, full_y_train)

final_preds = final_lr.predict(X_public_scaled_full)
final_public_f1 = f1_score(public_test_clean['Converted'], final_preds)

print(f"100% Train Data -> Public Test F1-Score: {final_public_f1:.4f}")

100% Train Data -> Public Test F1-Score: 0.3986


In [48]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Combine training and validation back into the 100% full train dataset
full_train_clean = pd.concat([X_train, X_val], axis=0).copy()
full_y_train = pd.concat([y_train, y_val], axis=0).copy()
public_test_clean = public_test.copy()

# 2. Add the missingness indicator column to both
full_train_clean['Time_On_Site_missing'] = full_train_clean['Time_On_Site'].isnull().astype(int)
public_test_clean['Time_On_Site_missing'] = public_test_clean['Time_On_Site'].isnull().astype(int)

# 3. Clean up missing values using the full training medians for all original numeric attributes
all_numeric_features = [
    'Age', 'Income', 'City_Tier', 'Pages_Viewed', 
    'Products_Viewed', 'Time_On_Site', 'Previous_Purchases', 
    'Discount_Seen', 'Time_On_Site_missing'
]

for col in all_numeric_features:
    median_val = full_train_clean[col].median()
    full_train_clean[col] = full_train_clean[col].fillna(median_val)
    public_test_clean[col] = public_test_clean[col].fillna(median_val)

# 4. Clamp continuous outliers at the 95th percentile of the full training data
for col in ['Time_On_Site', 'Income', 'Pages_Viewed']:
    upper_limit = full_train_clean[col].quantile(0.95)
    full_train_clean[col] = np.where(full_train_clean[col] > upper_limit, upper_limit, full_train_clean[col])
    public_test_clean[col] = np.where(public_test_clean[col] > upper_limit, upper_limit, public_test_clean[col])

# 5. Isolate and scale the features
scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(full_train_clean[all_numeric_features])
X_public_scaled_full = scaler_full.transform(public_test_clean[all_numeric_features])

# 6. Train Logistic Regression on 100% of data and test on public_test
final_lr = LogisticRegression(random_state=42, max_iter=1000)
final_lr.fit(X_train_scaled_full, full_y_train)

final_preds = final_lr.predict(X_public_scaled_full)
final_public_f1 = f1_score(public_test_clean['Converted'], final_preds)

print(f"100% Train Data (All Numeric Features) -> Public Test F1-Score: {final_public_f1:.4f}")

100% Train Data (All Numeric Features) -> Public Test F1-Score: 0.4020
